# 도로망 시계열 로더

폴더 구조:
```
도로망/
├── 2016/  01노드/  02링크/  03회전정보/
├── 2017/  01노드/  02링크/  03회전정보/
├── 2018/  01. 노드/  02. 링크/  03. 회전정보/
├── 2019/  01. 노드/  02. 링크/  03. 회전정보/
├── 2020/  2020-TM-GR-MR-LLV2 도로망(2019년 기준)/
├── 2021/  2021-TM-GR-MR-LLV2 도로망(2020년 기준)/
├── 2022/  2022-TM-GR-MR-LLV2 도로망(2021년 기준)/
├── 2023/  2023-TM-GR-MR-LLV2 도로망(2022년 기준)/
└── 2024/  01. 노드/  02. 링크/
```

### 연도별 스키마 변경 이력
| 연도 | 노드 변경 | 링크 변경 |
|------|-----------|----------|
| 2016 | FACILITY_ID, MAP_ID 있음 | ROAD_NO, BARRIER 있음 / ST_DIR·ED_DIR 없음 |
| 2017 | FACILITY_ID → TOLL_ID | ST_DIR·ED_DIR 추가, PAVEMENT 추가, ROAD_NO 삭제 |
| 2019 | - | PAVEMENT 삭제, FIRST_DO 추가 |
| 2021 | MAP_ID 삭제 | - |

In [2]:
import re
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [3]:
BASE_DIR = Path(r"C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT")
ROAD_DIR = BASE_DIR / "도로망"

# 아산시 충남 bounding box (EPSG:5179 기준 근사값 - 실제 shp 좌표계 확인 후 조정)
# WGS84: lon 126.8~127.1, lat 36.6~36.9
ASAN_BOUNDS_WGS84 = (126.8, 36.6, 127.1, 36.9)

In [4]:
# 노드/링크 컬럼 정규화 맵 (연도별 차이 흡수)
NODE_RENAME = {
    "FACILITY_ID": "TOLL_ID",   # 2016 → 통일
    "DIST_ID":     "DISTRICT_ID",
    "DIST_ID2":    "DISTRICT_ID2",
    "TRA_LIGHT":   "TRAFFIC_LIGHT",
}
LINK_RENAME = {
    "LINK_CATE":   "LINK_CATEGORY",
    "LANE":        "LANES",       # 2017~2018 shp 필드명 차이
    "UP_FROM_NO":  "UP_FROM_NODE",
    "DOWN_FROM_":  "DOWN_FROM_NODE",
    "UP_TO_NODE":  "UP_TO_NODE",
    "DOWN_TO_NO":  "DOWN_TO_NODE",
}

In [5]:
def find_shp(folder: Path, keyword: str) -> Path | None:
    """폴더 내 keyword 포함 .shp 파일 반환 (재귀)"""
    for shp in folder.rglob("*.shp"):
        if keyword.lower() in shp.stem.lower():
            return shp
    # keyword 매칭 실패 시 첫 번째 shp 반환
    shps = list(folder.rglob("*.shp"))
    return shps[0] if shps else None


def find_type_dir(year_dir: Path, keywords: list[str]) -> Path | None:
    """노드/링크/회전정보 하위 폴더 탐색"""
    for sub in sorted(year_dir.iterdir()):
        if not sub.is_dir():
            continue
        name = sub.name.lower().replace(" ", "").replace(".", "")
        if any(k in name for k in keywords):
            return sub
        # 2020~2023 패턴: 상위 폴더 하나 더 내려가기
        for subsub in sub.iterdir():
            if subsub.is_dir():
                n2 = subsub.name.lower().replace(" ", "").replace(".", "")
                if any(k in n2 for k in keywords):
                    return subsub
    return year_dir  # fallback: year_dir 자체에서 탐색


def load_shp(shp_path: Path, rename_map: dict, year: int) -> gpd.GeoDataFrame | None:
    if shp_path is None or not shp_path.exists():
        return None
    try:
        gdf = gpd.read_file(shp_path, engine="pyogrio")
        gdf.columns = [c.upper() for c in gdf.columns]
        gdf = gdf.rename(columns={k.upper(): v.upper() for k, v in rename_map.items()})
        gdf["data_year"] = year
        return gdf
    except Exception as e:
        print(f"  [ERR] {shp_path.name} ({year}): {e}")
        return None

In [6]:
# 폴더 구조 확인
print("발견된 연도 폴더:")
for year_dir in sorted(ROAD_DIR.iterdir()):
    if not year_dir.is_dir():
        continue
    subs = [s.name for s in sorted(year_dir.iterdir()) if s.is_dir()]
    shps = list(year_dir.rglob("*.shp"))
    print(f"  {year_dir.name}/  하위폴더: {subs}  shp: {len(shps)}개")

발견된 연도 폴더:
  2016/  하위폴더: ['01노드', '02링크', '03회전정보']  shp: 3개
  2017/  하위폴더: ['01노드', '02링크', '03회전정보']  shp: 3개
  2018/  하위폴더: ['01. 노드', '02. 링크', '03. 회전정보']  shp: 3개
  2019/  하위폴더: ['01. 노드', '02. 링크', '03. 회전정보']  shp: 3개
  2020/  하위폴더: ['2020-TM-GR-MR-LLV2 도로망(2019년 기준)']  shp: 3개
  2021/  하위폴더: ['2021-TM-GR-MR-LLV2 도로망(2020년 기준)']  shp: 3개
  2022/  하위폴더: ['2022-TM-GR-MR-LLV2 도로망(2021년 기준)']  shp: 3개
  2023/  하위폴더: ['2023-TM-GR-MR-LLV2 도로망(2022년 기준)']  shp: 3개
  2024/  하위폴더: ['01. 노드', '02. 링크']  shp: 2개
  _csv/  하위폴더: []  shp: 0개


In [7]:
# 전체 로드
nodes, links, turns = [], [], []

for year_dir in sorted(ROAD_DIR.iterdir()):
    if not year_dir.is_dir() or not year_dir.name.isdigit():
        continue
    year = int(year_dir.name)

    node_dir = find_type_dir(year_dir, ["노드", "node", "01노드", "01node"])
    link_dir = find_type_dir(year_dir, ["링크", "link", "02링크", "02link"])
    turn_dir = find_type_dir(year_dir, ["회전", "turn", "03회전"])

    node_shp = find_shp(node_dir, "node") if node_dir else None
    link_shp = find_shp(link_dir, "link") if link_dir else None
    turn_shp = find_shp(turn_dir, "turn") if turn_dir else None

    gdf_node = load_shp(node_shp, NODE_RENAME, year)
    gdf_link = load_shp(link_shp, LINK_RENAME, year)
    gdf_turn = load_shp(turn_shp, {},          year)

# 에러 발생 위치 (23번 라인 근처)
    if gdf_node is not None:
        # geometry 컬럼 설정
        if 'GEOMETRY' in gdf_node.columns:
            gdf_node = gdf_node.set_geometry('GEOMETRY')

        nodes.append(gdf_node)
        print(f"  [OK] {year} node  {len(gdf_node):>7,}행  crs={gdf_node.crs}")

    if gdf_link is not None:
        links.append(gdf_link)
        print(f"  [OK] {year} link  {len(gdf_link):>7,}행")
    if gdf_turn is not None:
        turns.append(gdf_turn)
        print(f"  [OK] {year} turn  {len(gdf_turn):>7,}행")

road = {
    "node": pd.concat(nodes, ignore_index=True) if nodes else None,
    "link": pd.concat(links, ignore_index=True) if links else None,
    "turn": pd.concat(turns, ignore_index=True) if turns else None,
}
print(f"\n로드 완료: {[k for k,v in road.items() if v is not None]}")

  [OK] 2016 node  427,695행  crs=PROJCS["Korea 2000 Katech(TM128)",GEOGCS["ITRF2000",DATUM["International_Terrestrial_Reference_Frame_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6656"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",128],PARAMETER["scale_factor",0.9999],PARAMETER["false_easting",400000],PARAMETER["false_northing",600000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
  [OK] 2016 link  551,277행
  [OK] 2016 turn  1,490,428행
  [OK] 2017 node  445,425행  crs=PROJCS["Korea 2000 Katech(TM128)",GEOGCS["ITRF2000",DATUM["International_Terrestrial_Reference_Frame_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6656"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARA

---
## 아산시 필터링

In [8]:
def filter_asan(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """좌표계 변환 후 아산시 bbox 클립"""
    if gdf.crs is None:
        print("  [WARN] CRS 없음 - EPSG:5179 가정")
        gdf = gdf.set_crs(epsg=5179)
    wgs = gdf.to_crs(epsg=4326)
    minx, miny, maxx, maxy = ASAN_BOUNDS_WGS84
    return wgs.cx[minx:maxx, miny:maxy].copy()

asan = {}
for name, gdf in road.items():
    if gdf is None:
        continue
    if hasattr(gdf, "geometry") and gdf.geometry.name in gdf.columns:
        asan[name] = filter_asan(gdf)
        print(f"  {name}: 전국 {len(gdf):,}행 → 아산시 {len(asan[name]):,}행")
    else:
        asan[name] = gdf  # 좌표 없으면 그대로

  node: 전국 4,245,473행 → 아산시 41,440행


In [9]:
# 연도별 아산시 도로 현황
if "link" in asan:
    summary = (
        asan["link"]
        .groupby("data_year")
        .agg(
            링크수=("LINK_ID", "count"),
            총연장_km=("LENGTH", lambda x: x.sum() / 1000),
            평균차로수=("LANES", "mean"),
        )
        .round(1)
    )
    print(summary.to_string())

              링크수  총연장_km  평균차로수
data_year                       
2016       551277   114.0    2.6
2017       574751   116.5    2.6
2018       605498   119.2    2.6
2019       610225   120.3    2.6
2020       614144   120.9    2.6
2021       620137   121.8    2.6
2022       625674   122.6    2.7
2023       627474   122.9    2.7
2024       628115   123.2    2.7


In [10]:
# 연도별 도로등급 분포
ROAD_RANK_LABEL = {
    "101": "고속국도", "102": "도시고속", "103": "일반국도",
    "104": "특별시도", "105": "광역시도", "106": "국가지원지방도",
    "107": "지방도",   "108": "시도",     "109": "군도",
    "110": "구도",
}
if "link" in asan and "ROAD_RANK" in asan["link"].columns:
    asan["link"]["등급명"] = asan["link"]["ROAD_RANK"].map(ROAD_RANK_LABEL).fillna("기타")
    print(asan["link"].groupby(["data_year", "등급명"])["LINK_ID"].count().unstack(fill_value=0).to_string())

등급명         고속국도   광역시도  국가지원지방도      기타  도시고속    시도   일반국도     지방도    특별시도
data_year                                                                  
2016        9509  13242    39791       0  1635  6683  56680  331204   92533
2017       11194  13969    41102       0  1715  7096  58596  345451   95628
2018       13420  14522    42219       0  1866  7685  60308  362931  102547
2019           0      0        0  610225     0     0      0       0       0
2020           0      0        0  614144     0     0      0       0       0
2021           0      0        0  620137     0     0      0       0       0
2022       13477  15115    44086       0  1861  8022  62418  375808  104887
2023       13917  15145    44210       0  1845  8232  62510  376570  105045
2024       13986  15175    44254       0  1845  8301  62791  376540  105223


---
## CSV 저장 (CP949)

In [11]:
SAVE_DIR = ROAD_DIR / "_csv"
SAVE_DIR.mkdir(exist_ok=True)

for name, gdf in asan.items():
    if gdf is None:
        continue
    # geometry 컬럼 제거 후 CSV 저장
    df = gdf.drop(columns=[gdf.geometry.name], errors="ignore") if hasattr(gdf, "geometry") else gdf
    out = SAVE_DIR / f"road_{name}_asan.csv"
    try:
        df.to_csv(out, index=False, encoding="cp949")
        print(f"[저장] {out.name}  {df.shape}")
    except UnicodeEncodeError:
        out_utf = SAVE_DIR / f"road_{name}_asan_utf8.csv"
        df.to_csv(out_utf, index=False, encoding="utf-8-sig")
        print(f"[fallback utf8] {out_utf.name}  {df.shape}")

# 전국 데이터도 저장 (용량 주의)
save_all = input("전국 데이터도 저장? (y/n): ").strip().lower() == "y"
if save_all:
    for name, gdf in road.items():
        if gdf is None:
            continue
        df = gdf.drop(columns=[gdf.geometry.name], errors="ignore") if hasattr(gdf, "geometry") else gdf
        out = SAVE_DIR / f"road_{name}_all.csv"
        try:
            df.to_csv(out, index=False, encoding="cp949")
            print(f"[저장] {out.name}  {df.shape}")
        except UnicodeEncodeError:
            out_utf = SAVE_DIR / f"road_{name}_all_utf8.csv"
            df.to_csv(out_utf, index=False, encoding="utf-8-sig")
            print(f"[fallback utf8] {out_utf.name}  {df.shape}")

print(f"\n저장 경로: {SAVE_DIR}")

[저장] road_node_asan.csv  (41440, 14)
[저장] road_link_asan.csv  (5457295, 41)
[저장] road_turn_asan.csv  (9441734, 19)
[저장] road_node_all.csv  (4245473, 14)
[저장] road_link_all.csv  (5457295, 41)
[저장] road_turn_all.csv  (9441734, 19)

저장 경로: C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT\도로망\_csv
